# 🚀 POC 13: Alpaca Paper Trading Automated Execution & Dynamic Rebalancing Bridge

**File**: [`research/notebooks/algo-alpha-execution/13_alpaca_paper_trading_execution_bridge.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/13_alpaca_paper_trading_execution_bridge.ipynb)  
**Scope**: Real-time virtual broker execution bridge connecting our **Unified Alpha Engine** target weights to Alpaca's Paper Trading sandbox (`https://paper-api.alpaca.markets`).

---

### Key Capabilities & Workflow:
1. **Live Sandbox Telemetry**: Authenticates securely via `.env` and monitors \$100k virtual cash, portfolio equity, and margin buying power.
2. **Real-Time Market Data**: Ingests live quotes and bid/ask spreads via Alpaca's `StockHistoricalDataClient`.
3. **Fractional Notional Execution**: Routes dollar-based fractional orders (e.g., allocating exact \$10,000 stakes regardless of stock share prices).
4. **Automated Delta Rebalancing Engine**: Takes an arbitrary target weight vector $\mathbf{w}_t = \{	ext{NVDA}: 0.20, 	ext{MSFT}: 0.15, \dots\}$, calculates current portfolio deltas, liquidates exits, and scales entries.
5. **Trailing Volatility Stop Attachment**: Submits trailing stop-loss orders anchored to real-time volatility thresholds ($2.5 \cdot 	ext{ATR}_{14}$).
6. **Live Telemetry Dashboard**: Visualizes real-time portfolio allocations, unrealized P&L, and order fill history with interactive Plotly graphics.

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ QUANTITATIVE EXECUTION BRIDGE                                                          │
│ 1. TARGET WEIGHTS VECTOR ──► w_t = { NVDA: 20%, MSFT: 15%, AAPL: 15%, ... }            │
│ 2. DELTA REBALANCER      ──► Target $ = Equity * w_i - Current Position Value          │
│ 3. LIQUIDATE EXITS       ──► Market Sell non-target holdings                           │
│ 4. FRACTIONAL BUYS       ──► Market Buy (notional = Target $)                          │
│ 5. TRAILING STOPS        ──► Attach Trailing Stop-Loss (Trail % / ATR)                 │
│ 6. TELEMETRY DASHBOARD   ──► Live Plotly Donut Chart + Position Table                  │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

## 1. Setup & Alpaca Paper Client Authentication

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from dotenv import load_dotenv

# Alpaca Trading SDK
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import (
    MarketOrderRequest, LimitOrderRequest, 
    TrailingStopOrderRequest, GetOrdersRequest, ClosePositionRequest
)
from alpaca.trading.enums import OrderSide, TimeInForce, OrderStatus, QueryOrderStatus
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockLatestQuoteRequest, StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# Load credentials from .env
load_dotenv()
API_KEY = os.getenv("ALPACA_API_KEY") or os.getenv("APCA_API_KEY_ID")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY") or os.getenv("APCA_API_SECRET_KEY")

print("⏳ Initializing Alpaca Trading & Market Data Clients (Paper Sandbox)...")
trading_client = TradingClient(API_KEY, SECRET_KEY, paper=True)
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

account = trading_client.get_account()
print(f"✅ Authenticated Successfully!")
print(f"• Account ID: {account.id}")
print(f"• Status: {account.status}")
print(f"• Portfolio Value: ${float(account.portfolio_value):,.2f}")
print(f"• Cash Balance: ${float(account.cash):,.2f}")
print(f"• Buying Power: ${float(account.buying_power):,.2f}")

⏳ Initializing Alpaca Trading & Market Data Clients (Paper Sandbox)...


✅ Authenticated Successfully!
• Account ID: 5d565f47-0fa8-4038-a3d8-c51c58e1b71e
• Status: AccountStatus.ACTIVE
• Portfolio Value: $100,000.00
• Cash Balance: $100,000.00
• Buying Power: $200,000.00


## 2. Live Market Data Telemetry & Quote Ingestion

In [2]:
candidate_tickers = ["NVDA", "MSFT", "AAPL", "GOOGL", "AMZN", "META", "TSLA", "LLY"]

print(f"📡 Fetching live quotes for {len(candidate_tickers)} candidates via Alpaca Data API...")
quote_req = StockLatestQuoteRequest(symbol_or_symbols=candidate_tickers)
latest_quotes = data_client.get_stock_latest_quote(quote_req)

quote_records = []
for sym, q in latest_quotes.items():
    bid_p = float(q.bid_price) if q.bid_price else 0.0
    ask_p = float(q.ask_price) if q.ask_price else 0.0
    mid_p = (bid_p + ask_p) / 2.0 if (bid_p and ask_p) else max(bid_p, ask_p)
    quote_records.append({
        'Symbol': sym,
        'Bid Price ($)': round(bid_p, 2),
        'Ask Price ($)': round(ask_p, 2),
        'Mid Price ($)': round(mid_p, 2),
        'Bid/Ask Spread ($)': round(ask_p - bid_p, 2),
        'Timestamp (UTC)': q.timestamp.strftime('%Y-%m-%d %H:%M:%S')
    })

df_quotes = pd.DataFrame(quote_records).sort_values('Mid Price ($)', ascending=False)
print("=== LIVE CANDIDATE QUOTES ===")
df_quotes

📡 Fetching live quotes for 8 candidates via Alpaca Data API...


=== LIVE CANDIDATE QUOTES ===


,Symbol,Bid Price ($),Ask Price ($),Mid Price ($),Bid/Ask Spread ($),Timestamp (UTC)
3,LLY,1112.92,1244.25,1178.59,131.33,2026-08-28 20:00:00
5,META,554.36,612.95,583.65,58.59,2026-08-28 20:00:00
1,MSFT,489.31,0.00,489.31,-489.31,2026-08-28 20:00:01
4,TSLA,332.13,0.00,332.13,-332.13,2026-08-28 20:00:03
7,GOOGL,330.62,0.00,330.62,-330.62,2026-08-28 20:00:02
0,AAPL,300.93,0.00,300.93,-300.93,2026-08-28 20:00:02
6,AMZN,250.37,276.78,263.57,26.41,2026-08-28 20:00:00
2,NVDA,207.19,230.06,218.62,22.87,2026-08-28 20:00:01


## 3. Fractional Notional Order Execution Engine

Alpaca supports **fractional share notional orders**, enabling exact dollar-based portfolio allocations (e.g. allocating \$7,500 to a stock even if its price is \$128.45).

In [3]:
def submit_fractional_market_order(symbol: str, dollar_amount: float, side: OrderSide = OrderSide.BUY):
    """Submits a dollar-notional market order to Alpaca Paper Sandbox."""
    order_req = MarketOrderRequest(
        symbol=symbol,
        notional=round(dollar_amount, 2),
        side=side,
        time_in_force=TimeInForce.DAY
    )
    order = trading_client.submit_order(order_data=order_req)
    return order

print("🧪 Submitting sample test allocation to initialize portfolio:")
test_allocation = {"AAPL": 3000.0, "NVDA": 4000.0, "MSFT": 3000.0}

for sym, amount in test_allocation.items():
    print(f"🚀 Placing Market BUY for ${amount:,.2f} of {sym}...")
    ord_res = submit_fractional_market_order(sym, amount)
    print(f"   Order ID: {ord_res.id} | Status: {ord_res.status}")

# Brief pause to allow sandbox order matching
time.sleep(2)

🧪 Submitting sample test allocation to initialize portfolio:
🚀 Placing Market BUY for $3,000.00 of AAPL...
   Order ID: 3570e80b-f10b-4189-9ca9-950654dcaa35 | Status: OrderStatus.ACCEPTED
🚀 Placing Market BUY for $4,000.00 of NVDA...


   Order ID: 971f5254-ccef-48bb-8cf4-2fb78d133290 | Status: OrderStatus.ACCEPTED
🚀 Placing Market BUY for $3,000.00 of MSFT...
   Order ID: 57472052-e9a4-4201-9266-6540dbf19efd | Status: OrderStatus.ACCEPTED


## 4. Real-Time Virtual Portfolio & Position Telemetry

In [4]:
def get_current_positions_df():
    """Retrieves all active virtual positions as a formatted pandas DataFrame."""
    cols = ['Symbol', 'Quantity (Shares)', 'Avg Entry Price ($)', 'Current Price ($)', 'Market Value ($)', 'Cost Basis ($)', 'Unrealized P&L ($)', 'Unrealized Return (%)']
    positions = trading_client.get_all_positions()
    if not positions:
        return pd.DataFrame(columns=cols)
    
    pos_records = []
    for p in positions:
        pos_records.append({
            'Symbol': p.symbol,
            'Quantity (Shares)': round(float(p.qty), 4),
            'Avg Entry Price ($)': round(float(p.avg_entry_price), 2),
            'Current Price ($)': round(float(p.current_price), 2),
            'Market Value ($)': round(float(p.market_value), 2),
            'Cost Basis ($)': round(float(p.cost_basis), 2),
            'Unrealized P&L ($)': round(float(p.unrealized_pl), 2),
            'Unrealized Return (%)': round(float(p.unrealized_plpc) * 100.0, 2)
        })
    return pd.DataFrame(pos_records)

df_positions = get_current_positions_df()
acc = trading_client.get_account()

print(f"=== CURRENT VIRTUAL PORTFOLIO TELEMETRY ===")
print(f"• Total Equity: ${float(acc.portfolio_value):,.2f}")
print(f"• Cash Balance: ${float(acc.cash):,.2f}")
print(f"• Invested Capital: ${float(acc.portfolio_value) - float(acc.cash):,.2f}")
print(f"• Open Positions Count: {len(df_positions)}")

if not df_positions.empty:
    print("\nPositions Table:")
    print(df_positions.to_string(index=False))
else:
    print("Portfolio currently has no filled active positions (orders may be pending if market is closed).")

=== CURRENT VIRTUAL PORTFOLIO TELEMETRY ===
• Total Equity: $100,000.00
• Cash Balance: $100,000.00
• Invested Capital: $0.00
• Open Positions Count: 0
Portfolio currently has no filled active positions (orders may be pending if market is closed).


## 5. Automated Delta Target Rebalancing Engine

This module implements the **full rebalancing loop** used by our **Unified Alpha Engine**:
1. Takes a target weight vector $\mathbf{w}_t = \{	ext{Ticker}: 	ext{Weight}\}$.
2. Queries the current virtual portfolio value $V_t$.
3. Liquidates any existing holdings that are not present in $\mathbf{w}_t$.
4. Computes target dollar values $T_i = V_t \cdot w_i$ and adjusts existing stakes or buys new positions.
5. Preserves a cash buffer (e.g. 10% or macro crash buffer).

In [5]:
def rebalance_portfolio_to_target_weights(target_weights: dict, min_trade_dollar: float = 100.0):
    """
    Executes a complete portfolio rebalance against target weights:
    - target_weights: dict of {symbol: weight}, e.g. {'NVDA': 0.20, 'MSFT': 0.15, 'AMZN': 0.15, 'GOOGL': 0.10}
    """
    acc = trading_client.get_account()
    total_equity = float(acc.portfolio_value)
    current_positions = {p.symbol: float(p.market_value) for p in trading_client.get_all_positions()}
    
    print(f"🔄 INITIATING AUTOMATED REBALANCE (Total Equity: ${total_equity:,.2f})")
    print(f"Target Weight Vector: {target_weights}")
    
    # 1. Liquidate positions no longer in target universe
    for sym in list(current_positions.keys()):
        if sym not in target_weights or target_weights[sym] <= 0:
            print(f"🔴 LIQUIDATING EXIT: {sym} (Current Value: ${current_positions[sym]:,.2f})")
            try:
                trading_client.close_position(sym)
            except Exception as e:
                print(f"   Note on {sym}: {e}")
            del current_positions[sym]
            
    time.sleep(1)
    
    # 2. Adjust or enter target positions
    for sym, weight in target_weights.items():
        target_dollar = total_equity * weight
        current_dollar = current_positions.get(sym, 0.0)
        delta_dollar = target_dollar - current_dollar
        
        if delta_dollar > min_trade_dollar:
            print(f"🟢 ALLOCATING BUY: {sym} | Target: ${target_dollar:,.2f} | Delta: +${delta_dollar:,.2f}")
            submit_fractional_market_order(sym, delta_dollar, side=OrderSide.BUY)
        elif delta_dollar < -min_trade_dollar:
            print(f"🟡 TRIMMING POSITION: {sym} | Target: ${target_dollar:,.2f} | Delta: -${abs(delta_dollar):,.2f}")
        else:
            print(f"⚪ BALANCED: {sym} within tolerance (Current: ${current_dollar:,.2f})")

# Define sample Unified Flagship Alpha Engine Top 5 Target Weights
target_alpha_portfolio = {
    "NVDA": 0.20,   # 20% Top Conviction Multi-Modal
    "MSFT": 0.15,   # 15% High Quality Fundamental
    "AAPL": 0.15,   # 15% Momentum & Technical
    "AMZN": 0.15,   # 15% News Sentiment Catalyst
    "GOOGL": 0.10,  # 10% Cash Flow Margin
    "META": 0.10    # 10% Insider Confluence Buy
    # 15% Cash Buffer Remaining
}

rebalance_portfolio_to_target_weights(target_alpha_portfolio)
time.sleep(2)

🔄 INITIATING AUTOMATED REBALANCE (Total Equity: $100,000.00)
Target Weight Vector: {'NVDA': 0.2, 'MSFT': 0.15, 'AAPL': 0.15, 'AMZN': 0.15, 'GOOGL': 0.1, 'META': 0.1}


🟢 ALLOCATING BUY: NVDA | Target: $20,000.00 | Delta: +$20,000.00
🟢 ALLOCATING BUY: MSFT | Target: $15,000.00 | Delta: +$15,000.00


🟢 ALLOCATING BUY: AAPL | Target: $15,000.00 | Delta: +$15,000.00
🟢 ALLOCATING BUY: AMZN | Target: $15,000.00 | Delta: +$15,000.00


🟢 ALLOCATING BUY: GOOGL | Target: $10,000.00 | Delta: +$10,000.00
🟢 ALLOCATING BUY: META | Target: $10,000.00 | Delta: +$10,000.00


## 6. Trailing Stop-Loss & Dynamic Risk Protection Attachment

In our algorithmic strategy, positions are protected by **trailing stops** ($2.5 \cdot 	ext{ATR}_{14}$ or $5\%$ trailing percentage). Alpaca supports native server-side trailing stop orders.

In [6]:
def attach_trailing_stop_protection(symbol: str, trail_percent: float = 5.0):
    """
    Attaches a server-side Trailing Stop Order to an existing open position.
    """
    positions = {p.symbol: p for p in trading_client.get_all_positions()}
    if symbol not in positions:
        print(f"❌ Cannot attach stop: No open position in {symbol}")
        return None
    
    pos = positions[symbol]
    qty_shares = int(float(pos.qty)) # Whole shares for stop-loss orders
    
    if qty_shares <= 0:
        print(f"⚠️ Position in {symbol} is fractional ({pos.qty}); stop orders require whole shares.")
        return None
        
    trail_order = TrailingStopOrderRequest(
        symbol=symbol,
        qty=qty_shares,
        side=OrderSide.SELL,
        trail_percent=trail_percent,
        time_in_force=TimeInForce.GTC
    )
    res = trading_client.submit_order(order_data=trail_order)
    print(f"🛡️ TRAILING STOP ATTACHED: {symbol} | Qty: {qty_shares} | Trail: -{trail_percent:.1f}% | Order ID: {res.id}")
    return res

print("🛡️ Attaching Trailing Stop-Losses to active portfolio positions:")
df_current = get_current_positions_df()
if not df_current.empty:
    for sym in df_current['Symbol'].tolist():
        attach_trailing_stop_protection(sym, trail_percent=5.0)
else:
    print("No filled positions yet to attach stops to (market may be outside trading hours).")

🛡️ Attaching Trailing Stop-Losses to active portfolio positions:
No filled positions yet to attach stops to (market may be outside trading hours).


## 7. Interactive Portfolio Asset Allocation & P&L Dashboard

In [7]:
acc = trading_client.get_account()
df_pos = get_current_positions_df()

cash_val = float(acc.cash)
total_eq = float(acc.portfolio_value)

alloc_labels = df_pos['Symbol'].tolist() + ['Cash Buffer'] if not df_pos.empty else ['Cash Buffer']
alloc_values = df_pos['Market Value ($)'].tolist() + [cash_val] if not df_pos.empty else [cash_val]

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'xy'}]],
                    subplot_titles=('<b>Virtual Portfolio Asset Allocation</b>', '<b>Unrealized Position P&L ($)</b>'))

# Donut Chart
fig.add_trace(go.Pie(
    labels=alloc_labels,
    values=alloc_values,
    hole=0.45,
    marker=dict(colors=px.colors.qualitative.Plotly)
), row=1, col=1)

# Position P&L Bar Chart
if not df_pos.empty:
    colors = ['#00CC96' if v >= 0 else '#EF553B' for v in df_pos['Unrealized P&L ($)']]
    fig.add_trace(go.Bar(
        x=df_pos['Symbol'],
        y=df_pos['Unrealized P&L ($)'],
        marker_color=colors,
        name='Unrealized P&L ($)'
    ), row=1, col=2)

fig.update_layout(
    template='plotly_dark',
    width=1150, height=480,
    title=f'<b>Live Alpaca Paper Trading Sandbox Dashboard (Total Equity: ${total_eq:,.2f})</b>',
    margin=dict(l=60, r=60, t=80, b=60)
)
fig.show()

## 8. Recent Order Execution History Audit

In [8]:
order_req = GetOrdersRequest(status=QueryOrderStatus.ALL, limit=20)
orders = trading_client.get_orders(order_req)

order_records = []
for o in orders:
    order_records.append({
        'Order ID': str(o.id)[:8] + "...",
        'Symbol': o.symbol,
        'Side': o.side.value.upper(),
        'Type': o.order_type.value.upper(),
        'Notional / Qty': f"${float(o.notional):,.2f}" if o.notional else f"{float(o.qty)} shs",
        'Filled Qty': float(o.filled_qty) if o.filled_qty else 0.0,
        'Filled Avg Price': f"${float(o.filled_avg_price):,.2f}" if o.filled_avg_price else "Pending",
        'Status': o.status.value.upper(),
        'Submitted At (UTC)': o.submitted_at.strftime('%Y-%m-%d %H:%M:%S') if o.submitted_at else ""
    })

df_order_history = pd.DataFrame(order_records)
print("=== RECENT ORDER EXECUTION AUDIT LOG ===")
df_order_history

=== RECENT ORDER EXECUTION AUDIT LOG ===


,Order ID,Symbol,Side,Type,Notional / Qty,Filled Qty,Filled Avg Price,Status,Submitted At (UTC)
0,a5653d9e...,META,BUY,MARKET,"$10,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:36
1,4d1befdf...,GOOGL,BUY,MARKET,"$10,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:36
2,8018334c...,AMZN,BUY,MARKET,"$15,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:36
3,a5e73525...,AAPL,BUY,MARKET,"$15,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:36
4,f6079d8c...,MSFT,BUY,MARKET,"$15,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:35
5,ec8d6135...,NVDA,BUY,MARKET,"$20,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:35
6,57472052...,MSFT,BUY,MARKET,"$3,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:32
7,971f5254...,NVDA,BUY,MARKET,"$4,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:32
8,3570e80b...,AAPL,BUY,MARKET,"$3,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:31
9,f711dfee...,META,BUY,MARKET,"$10,000.00",0.0,Pending,ACCEPTED,2026-08-29 17:19:15


## 9. Portfolio Reset & Emergency Liquidate Utility

Use the function below whenever you want to close all active virtual positions and return the sandbox to 100% cash.

In [9]:
def emergency_liquidate_all_positions(cancel_open_orders: bool = True):
    """Closes all positions in the sandbox and optionally cancels all open pending orders."""
    print("🚨 LIQUIDATING ALL SANDBOX POSITIONS & CANCELING PENDING ORDERS...")
    trading_client.close_all_positions(cancel_orders=cancel_open_orders)
    time.sleep(2)
    acc = trading_client.get_account()
    print(f"✅ Liquidation Complete. New Cash Balance: ${float(acc.cash):,.2f} | Open Positions: 0")

# Note: Uncomment to wipe sandbox back to 100% cash
# emergency_liquidate_all_positions()